# Class 3 — Going Outside: Acquiring External Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MostafaElFar96/QaF_Data_Analytics/blob/main/notebooks/Class_3_External_Data.ipynb)

Module 4 · Sanad Distribution case (fictional data). Run the cells top to bottom. Works in **Google Colab** (nothing to install) and in **VS Code** (Python + Jupyter extension).

## Setup

In [ ]:
# --- Setup: works in Google Colab and in VS Code -------------------------------
import os, sys, subprocess
if os.path.exists("../data/generate_sanad_data.py"):        # opened from the repo's notebooks/ folder
    os.chdir("..")
elif not os.path.exists("data/generate_sanad_data.py"):     # Colab: get the repo first
    if not os.path.exists("QaF_Data_Analytics"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/MostafaElFar96/QaF_Data_Analytics"], check=True)
    os.chdir("QaF_Data_Analytics")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "beautifulsoup4"], check=True)

if not os.path.exists("data/sanad/orders.csv"):            # same seed -> same data for everyone (~15 s)
    subprocess.run([sys.executable, "data/generate_sanad_data.py"], check=True)

import duckdb, pandas as pd
con = duckdb.connect("data/sanad.duckdb")

def run(statement):
    """Execute a statement (CREATE, INSERT...) and finish it."""
    con.execute(statement).fetchall()

def scalar(query):
    """Return the single value of a one-cell query."""
    return con.execute(query).fetchall()[0][0]
TABLES = ["areas", "retailers", "warehouses", "orders", "deliveries", "stock",
          "candidate_sites", "drive_times", "fsq_places_sample"]
existing = {r[0] for r in con.execute("SELECT table_name FROM information_schema.tables").fetchall()}
for t in TABLES:
    if t not in existing:
        run(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('data/sanad/{t}.csv')")

def sql(query):
    """Run SQL and return a pandas DataFrame (DuckDB speaks the same SQL as PostgreSQL here)."""
    return con.execute(query).df()

print("Ready:", ", ".join(f"{t} ({scalar(f'SELECT COUNT(*) FROM {t}'):,})" for t in TABLES[:6]))

## Video 2 — Scraping the practice listings site
The site is fictional and built for this course. The scraper reads `robots.txt` first and waits 2 seconds between pages (about 10 seconds in total). If the live site is unreachable, the cell serves the same pages from the repo on your machine.

In [ ]:
import time, requests, urllib.robotparser, threading, http.server, functools
from bs4 import BeautifulSoup

BASE = "https://mostafaelfar96.github.io/QaF_Data_Analytics"   # course practice site
try:
    requests.get(BASE + "/robots.txt", timeout=10).raise_for_status()
except Exception:
    class QuietHandler(http.server.SimpleHTTPRequestHandler):
        def log_message(self, *args):
            pass
    handler = functools.partial(QuietHandler, directory=os.getcwd())
    srv = http.server.ThreadingHTTPServer(("127.0.0.1", 8765), handler)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    BASE = "http://127.0.0.1:8765"
    print("Live site unreachable - using the local copy at", BASE)

UA = "CapstoneStudentBot/1.0 (student project)"
rp = urllib.robotparser.RobotFileParser(BASE + "/robots.txt")
rp.read()
print("Crawl-delay:", rp.crawl_delay(UA), "| contacts allowed?", rp.can_fetch(UA, BASE + "/contacts/WH-1001.html"))

rows = []
for page in range(1, 6):
    url = f"{BASE}/warehouses/page-{page}.html"
    if not rp.can_fetch(UA, url):
        break                                  # robots.txt says no
    html = requests.get(url, headers={"User-Agent": UA}, timeout=10).text
    for card in BeautifulSoup(html, "html.parser").select(".listing"):
        rows.append({
            "title":  card.select_one(".title").get_text(strip=True),
            "area":   card.select_one(".area").get_text(strip=True),
            "size":   card.select_one(".size").get_text(strip=True),
            "rent":   card.select_one(".price").get_text(strip=True),
            "truck":  card.select_one(".truck").get_text(strip=True),
            "zoning": card.select_one(".zoning").get_text(strip=True),
            "lat":    float(card["data-lat"]),
            "lon":    float(card["data-lon"]),
            "url":    card.select_one("a")["href"],
        })                                     # broker & phone NOT collected
    time.sleep(2)                              # rate limit: 1 request / 2 s
print(len(rows), "listings scraped")

## Video 2 — Drive times with the openrouteservice API
Get a free key at openrouteservice.org and paste it below. Without a key the notebook uses the course's reference drive times.

In [ ]:
ORS_KEY = ""   # paste your free key here (optional)
sites = sql("SELECT site_id, lon, lat FROM candidate_sites ORDER BY site_id")
nc_areas = sql("SELECT area_id, area_name, lon, lat FROM areas WHERE cluster = 'New Cairo' AND area_name <> 'Tagamoa 5' ORDER BY area_id")
if ORS_KEY:
    body = {
        "locations": sites[["lon", "lat"]].values.tolist() + nc_areas[["lon", "lat"]].values.tolist(),
        "sources": list(range(len(sites))),
        "destinations": list(range(len(sites), len(sites) + len(nc_areas))),
        "metrics": ["duration"],
    }
    r = requests.post("https://api.openrouteservice.org/v2/matrix/driving-hgv",
                      json=body, headers={"Authorization": ORS_KEY}, timeout=30)
    minutes = [[d / 60 for d in row] for row in r.json()["durations"]]
    free_flow = pd.DataFrame(minutes, index=sites.site_id, columns=nc_areas.area_name).round(1)
else:
    free_flow = sql("""SELECT t.site_id, a.area_name, t.minutes_free_flow FROM drive_times t JOIN areas a USING (area_id)
                       WHERE a.cluster = 'New Cairo' AND a.area_name <> 'Tagamoa 5'""").pivot(index="site_id", columns="area_name", values="minutes_free_flow")
    print("No API key: showing the course reference drive times (free-flow, minutes)")
free_flow

### Free-flow is not 10 am traffic
One way to adjust: a traffic factor = actual delivery times ÷ free-flow times. The course data uses 1.45.

In [ ]:
TRAFFIC_FACTOR = 1.45   # illustrative; estimate it from your own delivery times
(free_flow * TRAFFIC_FACTOR).round(1)

## Video 2 — Industrial zones from OpenStreetMap (Overpass API)
Needs internet. Be gentle with the free server: run it once.

In [ ]:
OVERPASS = """
[out:json][timeout:60];
area["name:en"="Cairo Governorate"]->.cairo;
way["landuse"="industrial"](area.cairo);
out center tags;
"""
try:
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": OVERPASS}, timeout=90,
                      headers={"User-Agent": UA})
    els = r.json()["elements"]
    zones = pd.DataFrame([{"osm_id": e["id"], "name": e.get("tags", {}).get("name", ""),
                           "lat": e["center"]["lat"], "lon": e["center"]["lon"]} for e in els])
    print(len(zones), "industrial areas found"); display(zones.head())
except Exception as e:
    print("Overpass not reachable right now:", e)

## Video 3 — Responsible collection: the source log

In [ ]:
source_log = pd.DataFrame([
    ("Sandbox Warehouse Listings", "scraping", "robots.txt read, crawl-delay 2 s, no broker data", "today"),
    ("openrouteservice",           "API",      "free key, daily quota respected",                 "today"),
    ("OpenStreetMap / Overpass",   "API",      "attribution, single request",                     "today"),
    ("City Population (CAPMAS)",   "download", "manual copy, source cited",                       "today"),
    ("Foursquare FSQ OS Places",   "open data","Apache 2.0, attribution",                         "today"),
], columns=["source", "method", "checks done", "retrieved on"])
source_log

## Video 4 — From raw listings to one comparison table

In [ ]:
df = pd.DataFrame(rows)
df["rent_egp"] = df["rent"].str.replace(r"[^\d]", "", regex=True).astype(int)
df["size_sqm"] = df["size"].str.replace(r"[^\d]", "", regex=True).astype(int)
df = df.drop_duplicates(subset=["title", "area", "size_sqm"])
df["source"] = "sandbox-warehouse-listings"
df["retrieved_at"] = pd.Timestamp.today().normalize()
run("CREATE OR REPLACE TABLE candidate_sites_scraped AS SELECT * FROM df")
print(len(rows), "scraped ->", len(df), "after removing duplicates")

In [ ]:
EAST = {"Suez Road", "Shorouk", "Katameya", "Badr City", "New Cairo", "Obour", "Cairo-Ismailia Road",
        "Madinaty", "El Rehab", "Nasr City", "Heliopolis"}
step1 = df[df.area.isin(EAST) & (df.size_sqm >= 2000)]
step2 = step1[step1.zoning.str.contains("Industrial") & step1.truck.str.endswith("12-ton trucks")]
step3 = step2[step2.rent_egp <= 470000]
funnel = pd.Series({"scraped": len(rows), "unique": len(df), "East & >= 2,000 sqm": len(step1),
                    "industrial & 12-ton access": len(step2), "near the rent cap": len(step3)})
print(funnel.to_string()); step3[["title", "area", "size_sqm", "rent_egp", "lat", "lon"]]

In [ ]:
# The two tables Class 4 starts from (reference versions shipped with the course data)
display(sql("SELECT * FROM candidate_sites"))
sql("SELECT * FROM drive_times LIMIT 10")

## Extra — Foursquare-style places (sample)
The course ships a **synthetic** sample shaped like Foursquare's open data (FSQ OS Places). The real dataset is a free download on Hugging Face (`foursquare/fsq-os-places`, Apache 2.0).

In [ ]:
sql("SELECT * FROM fsq_places_sample LIMIT 5")